In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Union
import numpy as np


ArrayLikeBytes = Union[bytes, bytearray, memoryview, np.ndarray]


@dataclass
class IQSignal:
    """
    Complex IQ Signal Container.

    Attributes
    ----------
    samples : np.ndarray
        Complex vector of IQ samples.
    sample_rate : float | None
        Optional sampling rate.
    metadata : dict | None
        Optional additional information.
    """
    samples: np.ndarray
    sample_rate: Optional[float] = None
    metadata: Optional[dict] = None

    def __post_init__(self):
        self.samples = np.asarray(self.samples, dtype=np.complex128)
        if self.samples.ndim != 1:
            raise ValueError("IQSignal.samples must be a 1D complex vector.")

    @property
    def n_samples(self) -> int:
        return self.samples.size

    @property
    def i(self) -> np.ndarray:
        return self.samples.real

    @property
    def q(self) -> np.ndarray:
        return self.samples.imag

    def copy(self) -> "IQSignal":
        return IQSignal(
            samples=self.samples.copy(),
            sample_rate=self.sample_rate,
            metadata=None if self.metadata is None else dict(self.metadata)
        )


@dataclass
class IQCompensationResult:
    """
    Detailed IQ Compensation Result.
    """
    corrected_signal: IQSignal
    mean_i: float
    mean_q: float
    power_i: float
    power_q: float
    gain_q: float
    cross_iq: float
    rho: float
    success: bool
    reason: str


class IQBufferLoader:
    """
    IQ Signal Loader from Raw Interleaved int8 Buffer:
    [I0, Q0, I1, Q1, ...]
    """

    @staticmethod
    def from_interleaved_int8(
        buffer: ArrayLikeBytes,
        sample_rate: Optional[float] = None,
        metadata: Optional[dict] = None
    ) -> IQSignal:
        """
        Converts an interleaved int8 buffer into a complex IQ signal.

        Parameters
        ----------
        buffer : bytes | bytearray | memoryview | np.ndarray
            Buffer in interleaved format [I0, Q0, I1, Q1, ...].
        sample_rate : float, optional
            Sampling rate.
        metadata : dict, optional
            Additional metadata.

        Returns
        -------
        IQSignal
            Complex IQ signal in np.complex128 format.
        """
        if buffer is None:
            raise ValueError("buffer cannot be None.")

        if isinstance(buffer, np.ndarray):
            raw = np.asarray(buffer, dtype=np.int8).ravel()
        else:
            raw = np.frombuffer(buffer, dtype=np.int8)

        if raw.size == 0:
            raise ValueError("empty buffer.")

        if raw.size % 2 != 0:
            raise ValueError(
                "The interleaved IQ buffer must have an even length: [I0, Q0, I1, Q1, ...]."
            )

        i_samples = raw[0::2].astype(np.float64)
        q_samples = raw[1::2].astype(np.float64)
        complex_samples = i_samples + 1j * q_samples

        return IQSignal(
            samples=complex_samples,
            sample_rate=sample_rate,
            metadata=metadata
        )


class IQCompensator:
    """
    Basic block-wise blind IQ imbalance compensator.

    Flow:
        1) DC removal in I and Q
        2) Gain balancing between branches
        3) Linear decorrelation of Q with respect to I
    """

    def __init__(self, eps: float = 1e-20):
        self.eps = float(eps)

    def compensate(self, signal: IQSignal, inplace: bool = False) -> IQCompensationResult:
        """
        Applies IQ compensation to a complex signal.

        Parameters
        ----------
        signal : IQSignal
            Input IQ signal.
        inplace : bool
            If True, modifies the original signal. If False, operates on a copy.

        Returns
        -------
        IQCompensationResult
            Result containing corrected signal and intermediate metrics.
        """
        if signal is None or signal.samples is None:
            return IQCompensationResult(
                corrected_signal=signal,
                mean_i=0.0,
                mean_q=0.0,
                power_i=0.0,
                power_q=0.0,
                gain_q=1.0,
                cross_iq=0.0,
                rho=0.0,
                success=False,
                reason="signal is None or contains no samples."
            )

        if signal.n_samples == 0:
            return IQCompensationResult(
                corrected_signal=signal,
                mean_i=0.0,
                mean_q=0.0,
                power_i=0.0,
                power_q=0.0,
                gain_q=1.0,
                cross_iq=0.0,
                rho=0.0,
                success=False,
                reason="The signal is empty."
            )

        work_signal = signal if inplace else signal.copy()

        x = work_signal.samples
        i = x.real
        q = x.imag

        # =====================================================
        # 1) DC Removal
        # =====================================================
        mean_i = float(np.mean(i))
        mean_q = float(np.mean(q))

        i_centered = i - mean_i
        q_centered = q - mean_q

        # =====================================================
        # 2) Power per Branch
        # =====================================================
        power_i = float(np.sum(i_centered * i_centered))
        power_q = float(np.sum(q_centered * q_centered))

        if power_i <= self.eps or power_q <= self.eps:
            work_signal.samples = i_centered + 1j * q_centered
            return IQCompensationResult(
                corrected_signal=work_signal,
                mean_i=mean_i,
                mean_q=mean_q,
                power_i=power_i,
                power_q=power_q,
                gain_q=1.0,
                cross_iq=0.0,
                rho=0.0,
                success=False,
                reason="Degenerate power in I or Q branch; stable compensation not possible."
            )

        # =====================================================
        # 3) Gain Balancing
        # =====================================================
        gain_q = float(np.sqrt(power_i / power_q))
        q_balanced = q_centered * gain_q

        # =====================================================
        # 4) Post-Balance Cross-Correlation
        # =====================================================
        cross_iq = float(np.sum(i_centered * q_balanced))

        # =====================================================
        # 5) Decorrelation Coefficient
        # =====================================================
        rho = float(cross_iq / (power_i + self.eps))

        # =====================================================
        # 6) Linear Decorrelation
        # =====================================================
        q_corrected = q_balanced - rho * i_centered

        work_signal.samples = i_centered + 1j * q_corrected

        return IQCompensationResult(
            corrected_signal=work_signal,
            mean_i=mean_i,
            mean_q=mean_q,
            power_i=power_i,
            power_q=power_q,
            gain_q=gain_q,
            cross_iq=cross_iq,
            rho=rho,
            success=True,
            reason="IQ compensation applied successfully."
        )